# Week 1

이번 주차에는 상품 리뷰를 숫자로 바꾸는 **데이터 전처리**를 진행합니다.  

[4주 흐름](../README.md)과 [실행 환경 안내](../docs/runtime.md)를 먼저 확인해주세요.

### 이번 주차의 학습 목표

- 평점과 리뷰로 이루어진 데이터의 구조 이해하기
- 사용할 데이터를 고르고 중복 제거하기
- 훈련 / 검증 / 테스트 데이터 나누기
- 리뷰와 정답을 숫자 배열로 만들기
- 문장 길이를 그래프로 확인하기

### 파일 구조

```text
gdg-7th-ai-mission/
├── week1/
│   ├── README.md
│   └── week1_mission.ipynb
├── docs/runtime.md
├── artifacts/  # 결과를 저장하면 생성됩니다.
│   └── week1/
│       └── week1_state.json
├── pyproject.toml
├── uv.lock
└── README.md
```


### Mission

이번 실습에서는 **함수의 틀과 복잡한 설정은 제공하고, 핵심 동작을 직접 작성**합니다.

- **[제공 코드]** 는 그대로 실행해주세요.
- **[미션]** 에서는 TODO 안내에 따라 ...를 코드로 바꿔주세요. label = None에서는 None을 정답을 구하는 조건식으로 바꿉니다.
- 각 단계의 **실행 결과 확인**에서 제공된 검토 코드를 실행하고, 출력이 예상과 맞는지 직접 확인해주세요.
- 코드를 수정했다면 해당 셀부터 아래로 다시 실행해주세요.


In [ ]:
# 로컬에서는 주석으로 둡니다. 웹 Colab에서만 필요한 줄을 해제해주세요.
# from google.colab import files
# files.upload()  # pyproject.toml 선택
# import sys
# print(sys.version)  # Python 3.12 확인
# assert sys.version_info[:2] == (3, 12), "Python 3.12 런타임을 선택해주세요."
# %pip install uv
# !uv pip install --python {sys.executable} -r /content/pyproject.toml
# 설치 후 Python 커널을 재시작하고 import 셀부터 실행해주세요.

# Drive에 결과를 보관할 때만 사용합니다.
# from google.colab import drive
# drive.mount('/content/drive')


In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# 프로젝트 루트나 week1 폴더에서 실행해도 결과는 같은 위치에 저장합니다.
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == 'week1' and (ROOT_DIR.parent / 'pyproject.toml').is_file():
    ROOT_DIR = ROOT_DIR.parent
SAVE_DIR = ROOT_DIR / 'artifacts' / 'week1'
# Drive를 연결했다면 아래 줄의 주석을 해제해주세요.
# SAVE_DIR = Path('/content/drive/MyDrive/gdg-mission/week1')
print('Python version:', sys.version)
print('TensorFlow version:', tf.__version__)
print('결과 저장 폴더:', SAVE_DIR)


Python version: 3.12.14 (main, Sep  1 2026, 14:17:39) [MSC v.1944 64 bit (AMD64)]
TensorFlow version: 2.21.0
결과 저장 폴더: C:\Users\a0103\Univ\gdg-ai-project\artifacts\week1


먼저 라이브러리를 불러왔습니다. NumPy는 배열을, TensorFlow는 텐서 연산을 다룹니다.  
Keras는 TensorFlow와 함께 사용할 레이어, 모델 API를 제공합니다.

- NumPy와 TensorFlow가 이번 실습에서 어떤 역할을 하는지 적어주세요.
- TensorFlow와 Keras의 관계를 적어주세요.

**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

#### 데이터 다운로드와 텍스트 읽기

네이버 쇼핑 리뷰를 불러와봅시다. 첫 300자를 출력해서 데이터의 구조를 살펴보세요.  
[데이터 설명](https://github.com/bab2min/corpus/tree/master/sentiment)


In [2]:
path_to_file = tf.keras.utils.get_file(
    'naver_shopping.txt',
    'https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt',
    cache_dir=str(Path.cwd() / 'data'), cache_subdir='',
)


20623547/20623547 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
text = Path(path_to_file).read_text(encoding='utf-8')
print('Length of text: {} characters'.format(len(text)))
print(text[:300])


Length of text: 8537926 characters
5	배공빠르고 굿
2	택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
5	아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 엉성하긴 하지만 편하고 가성비 최고예요.
2	선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전화했더니 바로주신다했지만 배송도 누락되어있었네요.. 확인안하고 바로 선물했으면 큰일날뻔했네요..이렇게 배송이 오래걸렸으면 사는거 다시 생각했을거같아요 아쉽네요..
5	민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ
2	비추합니다 계란 뒤


#### 1. 사용할 데이터 고르기

이 파일에는 한 줄에 리뷰 하나씩 들어 있습니다. 평점과 리뷰 사이는 탭으로 구분되어 있어요. 탭은 키보드의 Tab 키로 넣는 간격이고, 코드에서는 `\t`로 표시합니다.

```python
"5\t배송이 빨라요"  # 평점: 5, 리뷰: 배송이 빨라요
```

첫 줄부터 실제 리뷰가 들어 있으며, 3점 리뷰는 이미 제외되어 있습니다.

**이 단계는 제공 코드를 그대로 실행해주세요.** `filter_rows()`는 아래 조건에 맞는 원본 줄만 남깁니다.

- 탭으로 평점과 리뷰를 나눌 수 있고, 평점이 `1`, `2`, `4`, `5`인 줄
- 리뷰가 비어 있거나 공백뿐이지 않은 줄
- 완전히 같은 줄은 처음 나온 것 하나만 유지

리뷰 내용과 남은 줄의 순서는 바꾸지 않습니다. 결과는 원본 줄을 담은 리스트 `rows`예요.


In [ ]:
# [제공 코드] 사용할 줄을 고르고 중복을 제거합니다.
def filter_rows(text):
    rows = []
    seen = set()

    for row in text.splitlines():
        parts = row.split('\t', 1)
        if len(parts) != 2:
            continue

        rating, review = parts
        if rating not in ['1', '2', '4', '5'] or not review.strip():
            continue
        if row in seen:
            continue

        seen.add(row)
        rows.append(row)

    return rows


**실행 결과 확인**

아래 셀을 실행한 뒤, 사용할 리뷰 수가 원본 줄 수 이하인지 확인해주세요. 원본 데이터에 중복이나 잘못된 줄이 없다면 두 수가 같을 수도 있어요. 리뷰 예시에는 평점과 리뷰 내용이 함께 남아 있어야 합니다.


In [ ]:
# [검토 코드] 실행 결과를 위의 안내와 비교해주세요.
rows = filter_rows(text)
print('원본 줄 수:', len(text.splitlines()))
print('사용할 리뷰 수:', len(rows))
print('리뷰 예시:', rows[:3])


#### 2. 훈련 / 검증 / 테스트 데이터 나누기

리뷰를 세 묶음으로 나눠 각각 다른 용도로 사용합니다.

- **Train(훈련)**: 모델이 리뷰와 정답을 보며 학습하는 데 사용해요.
- **Validation(검증)**: 학습 도중 설정을 비교하고 선택하는 데 사용해요.
- **Test(테스트)**: 설정을 모두 정한 뒤 마지막으로 성능을 평가합니다.

이번 프로젝트에서는 **1~2점 리뷰는 부정으로, 4~5점 리뷰는 긍정으로 분류**합니다. 코드에서는 부정을 `0`, 긍정을 `1`로 표시해요. 두 종류를 구별하기 위해 붙인 이 정답을 **라벨**이라고 합니다.

**[미션] `train_test_split()`을 두 번 호출하는 코드를 작성해주세요.** 라벨 준비와 최종 반환 코드는 제공되어 있습니다.

1. 전체의 **30%를 테스트용**으로 나눕니다.
2. 남은 데이터의 **20%를 검증용**으로 나눠요. 나머지는 훈련용입니다.

100개라면 훈련용 56개, 검증용 14개, 테스트용 30개가 됩니다.

두 번 모두 `shuffle=True`, `random_state=seed`를 적용하고, `stratify`에는 나누려는 리뷰의 라벨을 전달해주세요. 리뷰와 라벨은 서로 짝을 유지하며 나뉩니다.

- 첫 번째 호출: `rows`, `labels`를 함께 나누고, 결과를 `remaining_rows`, `test_rows`, `remaining_labels`, `_` 순서로 받습니다. `_`에는 이후 사용하지 않을 테스트 라벨을 받아요.
- 두 번째 호출: `remaining_rows`를 나누고 `train_rows`, `validation_rows`로 받습니다. 비율 유지에는 `remaining_labels`를 사용해주세요.

**HINT:** `test_size`는 따로 떼어둘 비율입니다. 리뷰와 라벨을 함께 전달하면 결과도 ‘남은 리뷰, 떼어둔 리뷰, 남은 라벨, 떼어둔 라벨’ 순서로 네 개가 반환됩니다.


In [ ]:
# [미션] 데이터 분리 코드 두 곳을 작성해주세요.
def split_data(rows, seed=42):
    # 제공 코드: 부정은 0, 긍정은 1로 정답을 준비합니다.
    labels = [
        1 if row.split('\t', 1)[0] in ['4', '5'] else 0
        for row in rows
    ]

    # TODO ①: 전체의 30%를 테스트용으로 분리합니다.
    # 결과 변수: remaining_rows, test_rows, remaining_labels, _
    ...

    # TODO ②: 남은 데이터의 20%를 검증용으로 분리합니다.
    # 결과 변수: train_rows, validation_rows
    ...

    return train_rows, validation_rows, test_rows


**실행 결과 확인**

아래 셀을 실행하고, 세 묶음의 리뷰 수를 더하면 전체 리뷰 수와 같은지 확인해주세요. 출력 비율은 train 약 56%, validation 약 14%, test 약 30%가 되어야 합니다.


In [ ]:
# [검토 코드] 실행 결과를 위의 안내와 비교해주세요.
train_rows, validation_rows, test_rows = split_data(rows)

print('전체 리뷰 수:', len(rows))
for name, group in [('Train', train_rows), ('Validation', validation_rows), ('Test', test_rows)]:
    print(f'{name}: {len(group)}개 ({len(group) / len(rows):.1%})')
print('세 묶음의 합:', len(train_rows) + len(validation_rows) + len(test_rows))


**생각해보기:** 테스트용 데이터를 마지막 평가 전까지 따로 두는 이유는 무엇일까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

#### 3. Y(정답 라벨) 만들기

모델이 리뷰를 보고 긍정인지 부정인지 배우려면, 리뷰마다 **정답**이 필요합니다. 모델에 전달하는 정답 데이터를 **Y**라고 해요.

앞 단계에서는 데이터의 긍정·부정 비율을 유지하려고 라벨을 만들었어요. 이번에는 모델에 전달할 **정수 NumPy 배열**로 정리합니다. NumPy 배열은 여러 숫자를 담고 계산하는 데 사용하는 자료형이에요.

```text
"1\t상품이 망가졌어요" → 0 (부정)
"5\t배송이 빨라요"     → 1 (긍정)
```

**[미션] `label`에 정답을 담는 조건식을 작성해주세요.** 평점이 **4점 이상이면 긍정 `1`**, 그렇지 않으면 부정 `0`이 되도록 합니다. `if`/`else` 문을 여러 줄로 작성해도 괜찮아요. 숫자 `0`, `1`은 점수가 아니라 두 종류를 구별하는 표시예요.

반복문과 배열 변환은 제공 코드입니다. 입력된 리뷰 순서대로 정답을 만들고, `reshape(-1, 1)`로 한 열에 배치해요. `-1`은 데이터 개수에 맞게 행 수를 자동으로 정하라는 뜻입니다.

리뷰가 3개라면 배열은 아래처럼 **3행 1열**, 즉 `(3, 1)` 모양입니다. 이 모양을 `shape`라고 해요.

```python
[[0],
 [1],
 [1]]
```

리뷰가 없으면 제공 코드가 `(0, 1)` 모양의 배열을 만듭니다.


In [ ]:
# [미션] 평점을 긍정·부정 정답으로 바꾸는 부분을 작성해주세요.
def make_labels(rows):
    labels = []
    for row in rows:
        rating = int(row.split('\t', 1)[0])
        # TODO: None을 조건식으로 바꾸거나, 이 줄을 if/else 문으로 바꿔주세요.
        label = None
        labels.append(label)

    return np.array(labels, dtype=np.int64).reshape(-1, 1)


**실행 결과 확인**

아래 예시의 정답이 위에서부터 `0`, `1`, `1`인지 확인해주세요. 실제 훈련용 정답의 `shape`는 `(훈련용 리뷰 수, 1)`이어야 합니다.


In [ ]:
# [검토 코드] 실행 결과를 위의 안내와 비교해주세요.
print('예시 정답:')
print(make_labels(['1\t아쉬워요', '5\t좋아요', '4\t만족해요']))

train_Y = make_labels(train_rows)
validation_Y = make_labels(validation_rows)
test_Y = make_labels(test_rows)

print('train_Y shape:', train_Y.shape)
print('정답에 사용된 값:', np.unique(train_Y))
print('train_Y sample:', train_Y[:5])


#### 4. 리뷰 문장 정제하기

이제 평점을 제외한 **리뷰 문장**을 다뤄볼게요. 리뷰는 모델이 읽을 입력 데이터 **X**로 사용합니다.

리뷰에는 특수문자, 이모지, 불필요한 공백이 섞여 있어요. 이번 실습에서는 아래 규칙으로 문장을 정리합니다. 이런 과정을 **정제**라고 해요.

- 한글, 영문, 숫자는 남깁니다. `ㄱ`, `ㅠ` 같은 한글 자음·모음도 포함해요.
- 나머지 문자와 이모지는 공백으로 바꿉니다.
- 영문은 소문자로 바꾸고, 연속 공백과 문장 앞뒤의 공백을 정리해요.

```text
정제 전: ' GOOD!!! 배송  빠름 😊 '
정제 후: 'good 배송 빠름'
```

**[미션] 소문자 변환과 공백 정리 코드를 작성해주세요.** 문자 선택 규칙은 제공되어 있습니다. `re.sub()`는 정해진 문자 규칙에 맞춰 내용을 바꾸는 함수예요.

- 영문을 소문자로 바꾼 결과를 `string`에 담습니다.
- 연속 공백을 하나로 줄이고 양끝 공백을 없앤 결과도 `string`에 담아주세요.

**HINT:** `lower()`, `split()`, `join()`을 조합해보세요. 인자 없이 `split()`을 호출하면 연속 공백과 양끝 공백을 제외하고 나눕니다. 모델 학습과 새 리뷰 예측에는 같은 `clean_str()`를 사용해요.


In [ ]:
# [미션] 각 TODO 아래에 문장 정리 코드를 작성해주세요.
def clean_str(string):
    # 제공 코드: 허용하지 않는 문자를 공백으로 바꿉니다.
    string = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣA-Za-z0-9]', ' ', string)

    # TODO ①: 영문을 소문자로 바꾸고 string에 담아주세요.
    ...

    # TODO ②: 연속 공백과 양끝 공백을 정리하고 string에 담아주세요.
    ...

    return string


**실행 결과 확인**

아래 셀의 정제 결과가 `'good 배송 빠름'`인지 확인해주세요. 이후 출력되는 리뷰 예시도 공백을 기준으로 나뉘어 있어야 합니다.


In [ ]:
# [검토 코드] 실행 결과를 위의 안내와 비교해주세요.
sample = ' GOOD!!! 배송  빠름 😊 '
print('정제 전:', repr(sample))
print('정제 후:', repr(clean_str(sample)))

train_text_X = [row.split('\t', 1)[1] for row in train_rows]
validation_text_X = [row.split('\t', 1)[1] for row in validation_rows]
test_text_X = [row.split('\t', 1)[1] for row in test_rows]

sentences = [clean_str(sentence).split() for sentence in train_text_X]
for sentence in sentences[:5]:
    print(sentence)


**생각해보기:** 이모지와 느낌표를 지우면 리뷰의 어떤 정보가 사라질 수 있을까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

#### 5. 텍스트를 숫자로 바꾸기

앞에서 정리한 리뷰 `"배송 정말 좋아요"`를 모델에 넣으려면 숫자로 바꿔야 합니다. 이 문장이 어떻게 바뀌는지 순서대로 살펴볼게요.

**① 훈련용 리뷰에서 단어를 모으고 번호를 정해요.**

훈련용 리뷰에 등장하는 단어를 모아 각각 번호를 붙입니다. 예를 들어 다음처럼 번호가 정해졌다고 해볼게요. 실제 번호는 훈련용 리뷰에 따라 달라집니다.

```text
배송 → 2
정말 → 3
좋아요 → 4
```

이 단어와 번호의 목록을 **단어 사전**이라고 해요. TensorFlow의 `TextVectorization`은 사전을 만들고 문장을 숫자로 바꿔주는 도구입니다. 코드에서는 이런 처리 도구를 **레이어**라고 불러요.

레이어의 `adapt()`에 훈련용 문장을 전달하면 단어와 등장 빈도를 살펴보고 사전을 준비합니다. **검증용·테스트용 리뷰는 사전을 만드는 데 사용하지 않아요.** 나중에 평가할 데이터를 미리 참고하지 않기 위해서입니다.

**② 문장의 각 단어를 사전에 있는 번호로 바꿔요.**

문장을 공백으로 나누고, 각 단어를 사전에 있는 번호로 바꿉니다. 이때 나눈 단위를 **토큰**이라고 해요.

사전에 없는 단어는 번호를 찾을 수 없으므로 `1`로 표시합니다. 이런 단어를 **OOV(Out-Of-Vocabulary)** 라고 해요.

```text
"배송 처음보는단어" → [2, 1]
```

문장을 변환할 때는 **이미 준비한 사전**을 사용합니다. 훈련용·검증용·테스트용 문장 모두 같은 번호를 써야 하므로, 변환할 때마다 `adapt()`를 다시 실행하지 않아요.

**③ 문장마다 숫자 개수를 맞춰요.**

문장마다 길이가 다르므로, 이번에는 출력 길이를 5로 정했다고 해볼게요. 숫자가 부족한 자리는 `0`으로 채웁니다. 이를 **패딩**이라고 해요.

```text
"배송 정말 좋아요" → [2, 3, 4] → [2, 3, 4, 0, 0]
"배송 좋아요"      → [2, 4]    → [2, 4, 0, 0, 0]
```

길이가 5를 넘는 문장은 앞의 토큰 5개만 남기고 뒤를 자릅니다. 길이 맞추기는 레이어가 자동으로 처리해요. **실제 실습 코드에서는 `MAX_LEN = 25`를 사용합니다.**

변환 결과는 여러 문장의 숫자를 담은 **정수 Tensor**입니다. 여기서는 숫자 배열을 담는 자료형이라고 생각하면 돼요. 단어 변환에 쓰이는 `0`과 `1`은 정답 Y의 부정·긍정 표시와 용도가 다릅니다.

**이제 핵심 코드 두 곳을 작성해주세요.**

- **[미션 ①] `build_vectorizer()`**: 레이어 `layer`에 정제한 훈련용 문장 `cleaned_texts`를 전달해 사전을 준비해주세요. `adapt()`를 사용하는 한 줄을 작성합니다.
- **[미션 ②] `encode_texts()`**: 준비된 `vectorize_layer`로 `inputs`를 변환하고, 결과를 반환하는 한 줄을 작성해주세요. 이 함수에서는 `adapt()`를 사용하지 않습니다.

**HINT:** 레이어는 함수처럼 `레이어(입력)` 형태로 호출할 수 있어요. 문장 정제와 레이어 설정은 제공되어 있습니다.


In [ ]:
VOCAB_SIZE = 2000
MAX_LEN = 25


def build_vectorizer(train_texts, vocab_size=2000, max_len=25):
    cleaned_texts = [clean_str(sentence) for sentence in train_texts]
    layer = tf.keras.layers.TextVectorization(
        standardize=None,
        split='whitespace',
        max_tokens=vocab_size,
        output_mode='int',
        output_sequence_length=max_len,
    )

    # TODO ①: cleaned_texts로 layer의 단어 사전을 준비해주세요.
    ...

    return layer


def encode_texts(texts, vectorize_layer):
    cleaned_texts = [clean_str(sentence) for sentence in texts]
    inputs = tf.constant(cleaned_texts, dtype=tf.string)

    # TODO ②: vectorize_layer로 inputs를 변환한 결과를 반환해주세요.
    ...


**제공 코드 살펴보기**

| 설정 | 의미 |
| --- | --- |
| `standardize=None` | `clean_str()`로 정제하므로 추가 정제는 하지 않아요. |
| `split='whitespace'` | 공백을 기준으로 토큰을 나눕니다. |
| `output_mode='int'` | 토큰을 정수 번호로 바꿔요. |
| `max_tokens=vocab_size` | 사전의 최대 크기입니다. 예약된 번호 `0`, `1`도 포함해요. |
| `output_sequence_length=max_len` | 변환 결과의 길이입니다. 짧으면 `0`으로 채우고 길면 뒤를 잘라요. |

**실행 결과 확인**

아래 셀에서 `train_X`의 모양이 `(훈련용 리뷰 수, 25)`인지 확인해주세요. 짧은 문장 뒤에는 `0`이 채워져 있어요. 사전의 첫 두 항목은 빈 문자열 `''`과 사전에 없는 단어를 뜻하는 `'[UNK]'`입니다. 사전의 단어 번호는 출력 목록의 위치에 해당하며 0부터 시작해요.


In [ ]:
# [검토 코드] 실행 결과를 위의 안내와 비교해주세요.
vectorize_layer = build_vectorizer(train_text_X, VOCAB_SIZE, MAX_LEN)
train_X = encode_texts(train_text_X, vectorize_layer)
validation_X = encode_texts(validation_text_X, vectorize_layer)
test_X = encode_texts(test_text_X, vectorize_layer)

print('사전 앞 10개:', vectorize_layer.get_vocabulary()[:10])
print('train_X shape:', train_X.shape)
print('train_X sample:', train_X[:5])


**생각해보기:** 검증용·테스트용 문장을 변환할 때 사전을 새로 만들면 어떤 문제가 생길까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

#### 6. 문장 길이 시각화

앞 단계에서는 문장을 모두 `MAX_LEN = 25`에 맞췄습니다. 여기서 25는 글자 수가 아니라 **공백으로 나눈 토큰 수**예요. 더 긴 문장은 뒤쪽 내용이 잘립니다.

이번에는 **자르기 전 길이**를 확인해서 최대 길이 25로 얼마나 많은 리뷰를 담을 수 있는지 살펴봐요. 이미 길이가 맞춰진 `train_X`가 아니라 원래 리뷰 문장 `train_text_X`를 사용합니다.

**[미션 ①] 문장을 정제하고 공백으로 나눈 뒤, 토큰 수를 구하는 한 줄을 작성해주세요.** 계산한 결과는 `length`에 담습니다. 각 문장의 길이를 리스트에 추가하는 코드는 제공되어 있어요.

```text
입력: ['배송!! 좋아요', '', '가격 정말 좋아요']
결과: [2, 0, 3]
```

**HINT:** `clean_str()`, `split()`, `len()`을 조합해보세요. 빈 문장은 토큰이 없으므로 길이가 0이에요.


In [ ]:
def token_lengths(texts):
    lengths = []
    for sentence in texts:
        # TODO ①: 정제한 문장의 토큰 수를 구해 length에 담아주세요.
        ...
        lengths.append(length)
    return lengths


**실행 결과 확인**

아래 예시의 결과가 `[2, 0, 3]`인지 확인해주세요.


In [ ]:
# [검토 코드] 실행 결과를 위의 안내와 비교해주세요.
print(token_lengths(['배송!! 좋아요', '', '가격 정말 좋아요']))


**② 문장 길이를 그래프로 그려봐요.**

훈련용 문장의 길이를 구하고 짧은 순서대로 정렬하는 코드는 제공되어 있습니다. 이를 **오름차순 정렬**이라고 해요.

**[미션 ②] 그래프에 x축과 y축 이름을 넣는 코드를 작성해주세요.** 영문으로 적어도 괜찮아요. `plt.xlabel()`과 `plt.ylabel()`을 사용할 수 있습니다.

- x축: 길이순으로 정렬된 문장의 순서입니다. 예: `Sentence index`
- y축: 문장의 토큰 수입니다. 예: `Token count`

**실행 결과 확인:** 그래프는 오른쪽으로 갈수록 길이가 같거나 커져야 합니다. y값이 25를 넘는 부분은 뒤쪽 내용이 잘리는 문장이에요.

아래 출력의 비율이 `90.0%`라면, 훈련용 문장의 90%를 자르지 않고 담을 수 있다는 뜻입니다.


In [ ]:
sentence_len = sorted(token_lengths(train_text_X))

plt.plot(sentence_len)
# TODO ②: x축과 y축 이름을 넣어주세요.
...
plt.show()

lengths = np.array(sentence_len)
print('MAX_LEN 이하 문장 수:', int(np.sum(lengths <= MAX_LEN)))
print(f'MAX_LEN 이하 비율: {np.mean(lengths <= MAX_LEN):.1%}')


**생각해보기:** 출력된 비율을 근거로 `MAX_LEN = 25`가 적절한지 설명해주세요. 길이를 더 짧게 또는 길게 설정하면 보존되는 내용과 숫자 배열의 크기는 어떻게 달라질까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

#### 7. 전처리 결과 저장과 복원

지금까지 만든 변수와 함수는 커널을 종료하면 메모리에서 사라집니다. **커널은 노트북의 코드를 실행하는 프로그램**이에요. 노트북을 저장해도 실행 중인 변수까지 자동으로 보관되는 것은 아닙니다.

커널을 다시 시작해도 작업 결과를 불러올 수 있도록 필요한 결과를 별도 파일로 저장합니다. **이 단계는 제공 코드를 그대로 실행하고 결과를 확인해주세요.**

**① 결과를 파일로 저장해요.**

아래 코드는 `week1_state.json`에 다음 내용을 기록합니다. JSON은 항목 이름과 값을 텍스트로 저장하는 파일 형식이에요.

| 저장 항목 | 용도 |
| --- | --- |
| 훈련·검증·테스트용 원본 줄 | 분리한 원본 데이터를 그대로 복원해요. |
| 단어 사전 | 단어마다 부여한 번호를 유지합니다. |
| 사전 크기와 최대 문장 길이 | 같은 설정으로 숫자 변환을 진행해요. |
| `seed` | 데이터 분리에 사용한 기준값을 기록합니다. |

숫자 배열 `train_X`, `train_Y` 자체를 저장하는 것은 아닙니다. 저장한 원본 줄과 사전을 사용하면 다시 만들 수 있어요.

**실행 결과 확인:** 앞의 미션을 모두 실행한 뒤 저장 셀을 실행해주세요. 출력된 저장 위치에 JSON 파일이 생겼는지 확인하고, 완성한 노트북도 저장합니다. 기본 저장 폴더는 프로젝트 루트 아래의 `artifacts/week1`이에요. 프로젝트 루트나 `week1` 폴더에서 실행해도 같은 위치에 저장됩니다. Colab Web의 기본 작업 폴더에서는 `/content/artifacts/week1`을 사용해요.

완성한 노트북과 코드는 GitHub 레포에, 전처리 결과인 JSON은 Google Drive 또는 Colab 런타임에 보관해주세요.
저장과 복원 위치는 아래 코드의 `SAVE_DIR`에서 확인하고 필요하면 바꿔주세요.
Colab 런타임의 파일은 사라질 수 있으므로 계속 사용할 JSON은 Drive에 보관해주세요.


In [ ]:
# [제공 코드] 앞 단계의 실행 결과를 저장합니다.
state = dict(
    train_rows=train_rows,
    validation_rows=validation_rows,
    test_rows=test_rows,
    vocabulary=vectorize_layer.get_vocabulary(),
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    seed=42,
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
state_path = SAVE_DIR / 'week1_state.json'
state_path.write_text(json.dumps(state, ensure_ascii=False), encoding='utf-8')
print('저장 위치:', state_path.resolve())


**② 저장한 결과를 복원해요.**

아래 복원 셀은 저장한 사전과 설정으로 레이어를 만들고 새 리뷰를 변환합니다. 먼저 현재 커널에서 실행해본 뒤, 다음 순서로 새 커널에서도 확인해주세요.

1. 저장 셀 실행을 마치고 노트북을 저장합니다.
2. 상단의 **Restart(커널 재시작)** 를 실행해요. 실행 중인 변수와 함수가 초기화됩니다.
3. 라이브러리를 불러오고 `SAVE_DIR`를 설정하는 셀을 실행합니다.
4. 미션을 완성한 `clean_str()` 함수 정의 셀과 `build_vectorizer()`·`encode_texts()` 함수 정의 셀을 실행해주세요.
5. 바로 아래 복원 셀을 실행합니다. 데이터 다운로드, 사전 생성, 저장 셀은 다시 실행하지 않아도 돼요.

**실행 결과 확인:** 훈련·검증·테스트 리뷰 수가 저장 전과 같고, 새 리뷰의 숫자 변환 결과가 출력되는지 확인해주세요.

복원할 때는 **`adapt()`를 다시 호출하지 않습니다.** 저장된 사전으로 같은 단어 번호를 유지해야 하기 때문이에요. `clean_str()`와 `encode_texts()`도 저장 전과 같은 함수를 사용합니다.


In [ ]:
# [제공 코드] 저장한 결과를 읽고 새 리뷰를 숫자로 바꿉니다.
state_path = SAVE_DIR / 'week1_state.json'
if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
    restored_layer = tf.keras.layers.TextVectorization(
        standardize=None,
        split='whitespace',
        output_mode='int',
        max_tokens=state['vocab_size'],
        output_sequence_length=state['max_len'],
        vocabulary=state['vocabulary'],
    )
    print('복원한 집합 크기:', [len(state[name]) for name in ['train_rows', 'validation_rows', 'test_rows']])
    print('새 리뷰:', encode_texts(['배송!! 좋아요'], restored_layer))
else:
    print('저장 파일을 찾지 못했습니다. SAVE_DIR와 저장 셀 실행 여부를 확인해주세요.')


### 마무리

- `TODO`의 핵심 코드를 작성하고 위에서 아래로 실행해주세요.
- 제공된 검토 코드를 실행하고, ‘실행 결과 확인’의 안내와 비교해 결과를 직접 확인합니다.
- ‘생각해보기’에 코드와 출력 결과를 근거로 짧게 답변해주세요.
- 노트북과 JSON을 저장하고, 새 커널에서도 복원되는지 확인합니다.
- WIL에는 알게 된 점, 막힌 지점, 해결 방법을 300자 이상 작성해주세요.


### 선택 정리

이번 실습에서는 데이터 전처리에 공통 함수를 사용했습니다.  

데이터가 커지면 Tensorflow 의 `tf.data`의 배치 처리, 병렬 처리를 활용하거나,  
별도의 전처리 레이어 구성을 고려해볼 수 있습니다.

데이터 규모, 메모리, CPU/GPU, 저장/복원을 기준으로 어떤 방식이 맞을지 생각해봅시다.